In [43]:
from pathlib import Path
import numpy as np
import pandas as pd
import nd2
from ipywidgets import interact, IntSlider
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import interactive_output
import tifffile as tiff

# Metadata
RFP -> Exposuretime is 400ms
GFP -> Exposuretime is 60ms
BF -> Exposuretime is 60ms

to substract the background we are using .tif images, where the shutter of the camera is closed

In [ ]:
dark_400 = tiff.imread("/Volumes/TAYLOR-LAB/Huyen Anh /20221216 3nM 231-A8_NEMO_TRAF6_1um_38mol 001/20210531 Darkfield 400ms.tif")
dark_60 = tiff.imread("/Volumes/TAYLOR-LAB/Huyen Anh /20221216 3nM 231-A8_NEMO_TRAF6_1um_38mol 001/20210531 Darkfield 60ms.tif")

print(dark_400.shape)
print(dark_60.shape)

(600, 600)
(600, 600)


# showing the Cell Images
0 -> RFP
1 -> GFP
2 -> BF

In [ ]:
og_file = "/Volumes/TAYLOR-LAB/Huyen Anh /20221216 3nM 231-A8_NEMO_TRAF6_1um_38mol 001/20221216 3nM 231-A8_NEMO_TRAF6_1um_38mol 001.nd2"
my_array = nd2.imread(og_file)
my_file = nd2.ND2File(og_file)
my_file.shape

channel_names = ["Red", "GFP", "BF"]

def show_frame(t):

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    #[Red]
    axes[0].imshow(
        my_array[t, 0],
        cmap="magma",
        vmin=0,
        vmax=500
    )
    axes[0].set_title("mScarlet")
    axes[0].axis("off")

    #[GFP]
    axes[1].imshow(
        my_array[t, 1],
        cmap="viridis",
        vmin=0,
        vmax=200
    )
    axes[1].set_title("GFP")
    axes[1].axis("off")

    #[Brightfield]
    axes[2].imshow(
        my_array[t, 2],
        cmap="gray"
        
    )
    axes[2].set_title("BF")
    axes[2].axis("off")

    plt.suptitle(f"Timepoint {t}")
    plt.tight_layout()
    plt.show()

play = widgets.Play(
    value=0,
    min=0,
    max=my_array.shape[0]-1,
    step=1,
    interval=3,      # milliseconds between frames
)
slider = widgets.IntSlider(
    min=0,
    max=my_array.shape[0]-1,
    step=1,
    value=0,
    description="Time"
)

# Link play button to slider
widgets.jslink((play, 'value'), (slider, 'value'))

# Display
ui = widgets.HBox([play, slider])
out = interactive_output(show_frame, {'t': slider})

display(ui, out)

/var/folders/2q/qj12d6zd1qg4x5blqbp5z8fc0000gn/T/ipykernel_71468/3669268749.py:3: UserWarning: ND2File file not closed before garbage collection. Please use `with ND2File(...):` context or call `.close()`.
  my_file = nd2.ND2File(og_file)


Output()

# Backgroundsubstracting
substrating dark_frame with the associated channels

In [49]:
rfp_raw = my_array[:,0]
gfp_raw = my_array[:,1]
bf_raw = my_array[:,2]